# Decaying Windows

## Learning Objectives

1. **Motivate** the exponential decay model for recency weighting
2. **Derive** the update rule for a decaying sum
3. **Implement** decaying counts with a constant-time update
4. **Analyze** the effective window and memory requirements

## Why Decay?

Sliding windows have a cliff-edge problem: an element drops out of the window abruptly.

**Exponential decay** weights each element by how recent it is:
$$\hat{f}(t) = \sum_{i=0}^{t} (1-\lambda)^{t-i} \cdot x_i$$
where $\lambda \in (0,1)$ is the **decay factor** and $x_i$ is the stream value at time $i$.

Equivalently, with $c = 1 - \lambda$:
$$\hat{f}(t) = c \cdot \hat{f}(t-1) + x_t$$

**One-line constant-time update.** No need to remember any previous elements.

In [1]:
# Decaying sum: exact via recurrence
class DecaySum:
    def __init__(self, decay=0.001):
        # decay λ: contribution halves every log(2)/λ steps
        self.lam = decay
        self.c = 1.0 - decay
        self.S = 0.0
        self.t = 0
    def add(self, x=1):
        self.S = self.c * self.S + x
        self.t += 1
    def query(self): return self.S
    def half_life(self): return math.log(2)/self.lam

import math, random
rng = random.Random(3)

# Simulate a stream with two bursts
stream = []
for i in range(1000):
    # Burst at t=200 and t=800
    if 190 <= i < 210 or 790 <= i < 810:
        stream.append(10)
    else:
        stream.append(rng.randint(0,2))

dc = DecaySum(decay=0.005)
snapshots = []
for t, x in enumerate(stream):
    dc.add(x)
    if t % 100 == 99:
        snapshots.append((t+1, dc.query()))

print(f"Half-life: {dc.half_life():.1f} time steps")
print(f"""
{'Time':>6}  {'Decaying Sum':>14}""")
for t, v in snapshots:
    bar = '#' * int(v/5)
    print(f"{t:>6}  {v:>14.1f}  {bar}")

Half-life: 138.6 time steps

  Time    Decaying Sum
   100            90.2  ##################
   200           230.9  ##############################################
   300           274.5  ######################################################
   400           235.5  ###############################################
   500           224.1  ############################################
   600           223.7  ############################################
   700           220.3  ############################################
   800           307.1  #############################################################
   900           315.7  ###############################################################
  1000           254.8  ##################################################


## Effective Window

The decaying sum weights each past element by $c^{t-i} = (1-\lambda)^{t-i}$.

The **effective window** is the number of recent elements that contribute most of the mass:
$$N_{\text{eff}} = \frac{1}{1 - c} = \frac{1}{\lambda}$$
Because $\sum_{k=0}^{\infty} c^k = \frac{1}{1-c}$.

Elements older than $N_{\text{eff}}$ steps contribute less than $e^{-1} \approx 37\%$ of the total weight.

In [2]:
# Visualize weight decay
lambdas = [0.001, 0.005, 0.01, 0.05]
t_max = 500

print(f"{'λ':>8}  {'N_eff':>8}  {'Half-life':>10}  Weight at 2×N_eff")
for lam in lambdas:
    c = 1-lam
    n_eff = 1/lam
    hl = math.log(2)/lam
    weight_2x = c**(2*n_eff)
    print(f"{lam:>8.3f}  {n_eff:>8.0f}  {hl:>10.1f}  {weight_2x:.4f}")

       λ     N_eff   Half-life  Weight at 2×N_eff
   0.001      1000       693.1  0.1352
   0.005       200       138.6  0.1347
   0.010       100        69.3  0.1340
   0.050        20        13.9  0.1285


## Counting with Decay: Applications

### Decaying Count of Distinct Items

Maintain a hash map `counts[item]` where each item's count decays:
- On new item $x$ at time $t$: `counts[x] += 1`
- Periodically: multiply all values by $c$

**Problem:** hash map still grows. Fix: prune items with count < $\epsilon$.

### Decaying Heavy Hitters

Track items whose decayed frequency exceeds $\phi \cdot \hat{f}(t)$ (fraction of total decayed weight).

Possible with the **sticky sampling** or **SPACESAVING** algorithm adapted to decay.

### Decaying Histogram

Maintain histogram of quantiles with exponential decay on bin counts. Queries: `P(X ≤ x)` at any time.

In [3]:
# Decaying frequency table with pruning
class DecayFreqTable:
    def __init__(self, decay=0.005, prune_threshold=0.01):
        self.lam = decay
        self.c = 1.0 - decay
        self.counts = defaultdict(float)
        self.total = 0.0
        self.prune_t = prune_threshold
    def add(self, item):
        # Decay all counts lazily (amortize over adds)
        self.counts[item] = self.c * self.counts.get(item, 0.0) + 1.0
        self.total = self.c * self.total + 1.0
    def top_k(self, k=5):
        items = [(v, i) for i,v in self.counts.items()]
        items.sort(reverse=True)
        return [(i, v/self.total) for v,i in items[:k]]
    def prune(self):
        threshold = self.prune_t * self.total
        self.counts = defaultdict(float, {i:v for i,v in self.counts.items() if v >= threshold})

from collections import defaultdict
dft = DecayFreqTable(decay=0.005)
words = ['cat','dog','cat','bird','cat','dog','fish','cat','fish','dog']
extended_stream = words * 50 + ['cat']*200 + words[:5]*20

for t, w in enumerate(extended_stream):
    dft.add(w)
    if t % 200 == 199:
        dft.prune()

print("Top-5 items by decaying frequency:")
for item, freq in dft.top_k(5):
    bar = '#'*int(freq*50)
    print(f"  {item:>6}: {freq:.3f}  {bar}")


Top-5 items by decaying frequency:
     cat: 0.917  #############################################
     dog: 0.584  #############################
    fish: 0.402  ####################
    bird: 0.301  ###############
